# Quaderno Prefettizia preprocessing

This notebook loads the parsed `quaderno_prefettizia` MCQ JSONL file, filters relevant legal sections, cleans malformed PDF artifacts, deduplicates examples, checks overlap against existing processed Italian datasets, and saves an SFT-compatible JSONL file under:

`multilingual/data/processed/quaderno_prefettizia_clean/quaderno_prefettizia_clean.jsonl`

In [1]:
from pathlib import Path
import json
import re
from collections import Counter, defaultdict

import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

REPO_ROOT = Path.cwd()
while REPO_ROOT.name != "standard-project-m2-mystery-machine" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

RAW_PATH = REPO_ROOT / "multilingual/data/raw/quaderno_prefettizia/quaderno_prefettizia_parsed.jsonl"

OUTPUT_DIR = REPO_ROOT / "multilingual/data/processed/quaderno_prefettizia_clean"
OUTPUT_PATH = OUTPUT_DIR / "quaderno_prefettizia_clean.jsonl"
ECONOMIA_OUTPUT_PATH = OUTPUT_DIR / "quaderno_prefettizia_economia_politica_clean.jsonl"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MAIN_DATASET = "quaderno_prefettizia_clean"
ECONOMIA_DATASET = "quaderno_prefettizia_economia_politica_clean"

LANGUAGE = "italian"
SPLIT = "all"
VALID_ANSWERS = set("ABCDE")

KEEP_SECTION_PREFIXES = {
    "DIRITTO COMUNITARIO",
    "DIRITTO COSTITUZIONALE",
    "DIRITTO AMMINISTRATIVO",
    "DIRITTO CIVILE",
}

ECONOMIA_SECTION_PREFIXES = {
    "ECONOMIA POLITICA",
}

print("Repo root:", REPO_ROOT)
print("Raw path:", RAW_PATH)
print("Main output path:", OUTPUT_PATH)
print("Economia output path:", ECONOMIA_OUTPUT_PATH)

Repo root: /scratch/workspace/emma/standard-project-m2-mystery-machine
Raw path: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/raw/quaderno_prefettizia/quaderno_prefettizia_parsed.jsonl
Main output path: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/quaderno_prefettizia_clean/quaderno_prefettizia_clean.jsonl
Economia output path: /scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/quaderno_prefettizia_clean/quaderno_prefettizia_economia_politica_clean.jsonl


## Helpers

In [21]:
def read_jsonl(path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def remove_page_artifacts(text):
    text = clean_text(text)
    text = re.sub(r"\bPagina\s+\d+\b", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def write_jsonl(records, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + "\n")


def clean_text(x):
    if x is None or pd.isna(x):
        return ""
    text = str(x)
    text = text.replace("\u00a0", " ")
    text = text.replace("ﬁ", "fi").replace("ﬂ", "fl")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    return text.strip()


def normalize_for_dedup(text):
    text = clean_text(text).lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\w\sàèéìòóù]", "", text, flags=re.UNICODE)
    return text.strip()


# Optional helper; currently useful only if you later build IDs from section/source names.
def safe_id_part(x):
    x = clean_text(x)
    x = re.sub(r"\s+", "_", x)
    x = x.replace("/", "_").replace("\\", "_")
    x = re.sub(r"[^A-Za-z0-9_À-ÖØ-öø-ÿ-]", "", x)
    return x[:80]


def section_base(section):
    if section is None:
        return None
    section = clean_text(section).upper()
    section = section.split(" - DIFFICOLTA")[0].strip()
    return section


def section_is_kept(section):
    base = section_base(section)
    return base in KEEP_SECTION_PREFIXES

In [22]:
def extract_question_options(record):
    if "prompt" in record:
        question, options = parse_prompt(record["prompt"])
    else:
        question, options = "", {}

    if not question:
        for key in ["question", "domanda", "stem"]:
            if key in record:
                question = clean_question_text(record[key])
                break

    option_field_candidates = {
        "A": ["A", "option_a", "answer_a", "a"],
        "B": ["B", "option_b", "answer_b", "b"],
        "C": ["C", "option_c", "answer_c", "c"],
        "D": ["D", "option_d", "answer_d", "d"],
        "E": ["E", "option_e", "answer_e", "e"],
    }

    for label, keys in option_field_candidates.items():
        if label in options and options[label]:
            continue
        for key in keys:
            if key in record:
                value = clean_option_text(record[key])
                if value:
                    options[label] = value
                break

    if "options" in record:
        raw_options = record["options"]
        if isinstance(raw_options, dict):
            for label, value in raw_options.items():
                label = str(label).strip().upper()
                if label in VALID_ANSWERS and label not in options:
                    options[label] = clean_option_text(value)

        elif isinstance(raw_options, list):
            for idx, value in enumerate(raw_options):
                label = chr(ord("A") + idx)
                if label in VALID_ANSWERS and label not in options:
                    options[label] = clean_option_text(value)

    return question, dict(sorted(options.items()))


def get_answer(record):
    answer = clean_text(
        record.get("answer", record.get("correct_answer", record.get("label", "")))
    ).upper()
    answer = answer.replace(")", "").replace(".", "").strip()
    return answer


def get_section(record):
    metadata = record.get("metadata", {})
    section = metadata.get("section", record.get("section", None))
    if section is None:
        return None
    return clean_text(section).upper()


def make_prompt(question, options):
    labels = sorted(options.keys())
    option_lines = [f"{label}) {options[label]}" for label in labels]
    return f"{question}\n\n" + "\n".join(option_lines)


def row_is_malformed(question, options, answer):
    if not question or len(question) < 8:
        return True, "bad_question"

    if answer not in VALID_ANSWERS:
        return True, "invalid_answer"

    # Keep 3-option questions, but remove obviously broken A/B-only examples.
    if len(options) < 3:
        return True, "too_few_options"

    if answer not in options:
        return True, "answer_option_missing"

    for label, value in options.items():
        if label not in VALID_ANSWERS:
            return True, "invalid_option_label"

        if clean_text(value) in BAD_EXACT_OPTION_VALUES:
            return True, "bad_option_text"

        if len(clean_text(value)) < 2:
            return True, "option_too_short"

    option_values = [normalize_for_dedup(v) for v in options.values()]
    if len(option_values) != len(set(option_values)):
        return True, "duplicate_options"

    full_text = question + " " + " ".join(options.values())
    if full_text.count("�") >= 2:
        return True, "replacement_characters"

    return False, None


def validate_record(record, expected_dataset):
    assert set(record.keys()) >= {
        "id",
        "dataset",
        "language",
        "split",
        "task_type",
        "prompt",
        "answer",
        "metadata",
    }

    assert record["dataset"] == expected_dataset
    assert record["language"] == LANGUAGE
    assert record["split"] == SPLIT
    assert record["task_type"] == "mcq"
    assert record["answer"] in VALID_ANSWERS
    assert record["prompt"].strip()
    assert "\n\nA) " in record["prompt"]
    assert "\nB) " in record["prompt"]
    assert record["answer"] + ")" in record["prompt"]
    assert isinstance(record["metadata"]["source_idx"], int)

In [23]:
def section_is_economia_politica(section):
    base = section_base(section)
    return base in ECONOMIA_SECTION_PREFIXES


OPTION_LINE_RE = re.compile(
    r"(?m)^\s*([A-E])[\).]\s*(.*?)(?=^\s*[A-E][\).]\s*|\Z)",
    re.DOTALL,
)

BAD_EXACT_OPTION_VALUES = {"", ".", "-", "—", "–", "A", "I", "T"}


def remove_isolated_watermark_lines(text):
    text = clean_text(text)
    lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if stripped in {"A", "I", "T"}:
            continue
        lines.append(line)
    return clean_text("\n".join(lines))


def clean_option_text(text):
    text = remove_isolated_watermark_lines(text)
    text = remove_page_artifacts(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_question_text(text):
    text = remove_isolated_watermark_lines(text)
    text = remove_page_artifacts(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def parse_prompt(prompt):
    prompt = clean_text(prompt)
    matches = list(OPTION_LINE_RE.finditer(prompt))

    if not matches:
        return clean_question_text(prompt), {}

    question = prompt[:matches[0].start()].strip()
    options = {}

    for match in matches:
        label = match.group(1).strip().upper()
        value = clean_option_text(match.group(2))
        options[label] = value

    return clean_question_text(question), options

## Load

In [24]:
raw_records = read_jsonl(RAW_PATH)
df_raw = pd.DataFrame(raw_records)

print("Loaded rows:", len(df_raw))
print("Columns:", df_raw.columns.tolist())

sections = []
for record in raw_records:
    sections.append(get_section(record))

section_counts = pd.Series(sections).value_counts(dropna=False)

print("\nSection counts:")
display(section_counts.to_frame("n_examples"))

print("\nExample raw record:")
print(json.dumps(raw_records[0], ensure_ascii=False, indent=2)[:3000])

Loaded rows: 4525
Columns: ['id', 'dataset', 'language', 'task_type', 'prompt', 'answer', 'metadata']

Section counts:


,n_examples
ECONOMIA POLITICA - DIFFICOLTA' MEDIA,1333
DIRITTO CIVILE - DIFFICOLTA' ALTA,383
DIRITTO CIVILE - DIFFICOLTA' MEDIA,373
DIRITTO COSTITUZIONALE - DIFFICOLTA' MEDIA,368
DIRITTO COMUNITARIO - DIFFICOLTA' MEDIA,367
DIRITTO AMMINISTRATIVO - DIFFICOLTA' MEDIA,366
DIRITTO CIVILE - DIFFICOLTA' FACILE,226
DIRITTO COMUNITARIO - DIFFICOLTA' FACILE,223
DIRITTO COSTITUZIONALE - DIFFICOLTA' FACILE,222
DIRITTO AMMINISTRATIVO - DIFFICOLTA' FACILE,221



Example raw record:
{
  "id": "quaderno_prefettizia_italian_0",
  "dataset": "quaderno_prefettizia",
  "language": "italian",
  "task_type": "mcq",
  "prompt": "In materia di adesione all'unione Europea, secondo l'art. 49 TUE, un paese che intende aderire deve necessariamente:\n\nA) non essere coinvolto dalla procedura di crisi di governo\nB) aver abolito la pena di morte\nC) essere in democrazia da almeno 50 anni\nD) essere uno Stato europeo",
  "answer": "D",
  "metadata": {
    "source": "quaderno_prefettizia",
    "section": "DIRITTO COMUNITARIO - DIFFICOLTA' FACILE",
    "original_id": 1
  }
}


In [25]:
def build_clean_dataframe(raw_records, section_filter_fn, source_name):
    stats = Counter()
    clean_rows = []

    for source_idx, record in enumerate(raw_records):
        section = get_section(record)

        if not section_filter_fn(section):
            stats[f"removed_section:{section}"] += 1
            continue

        question, options = extract_question_options(record)
        answer = get_answer(record)

        bad, reason = row_is_malformed(question, options, answer)
        if bad:
            stats[f"removed_{reason}"] += 1
            continue

        prompt = make_prompt(question, options)

        metadata = dict(record.get("metadata", {}))
        metadata.update({
            "source": source_name,
            "section": section,
            "section_base": section_base(section),
            "original_id": record.get("id", record.get("original_id", source_idx)),
            "source_idx": source_idx,
            "n_options": len(options),
        })

        clean_rows.append({
            "source_idx": source_idx,
            "section": section,
            "section_base": section_base(section),
            "question": question,
            "options": options,
            "prompt": prompt,
            "answer": answer,
            "metadata": metadata,
        })

    df_out = pd.DataFrame(clean_rows)
    return df_out, stats

## Deduplicate

In [26]:
def deduplicate_dataframe(df_in):
    df_out = df_in.copy()
    df_out["dedup_key"] = df_out["prompt"].apply(normalize_for_dedup)

    labels_by_key = defaultdict(set)
    for _, row in df_out.iterrows():
        labels_by_key[row["dedup_key"]].add(row["answer"])

    conflicting_keys = {key for key, labels in labels_by_key.items() if len(labels) > 1}

    n_before = len(df_out)
    df_out = df_out[~df_out["dedup_key"].isin(conflicting_keys)].copy()
    n_after_conflicts = len(df_out)

    df_out = df_out.drop_duplicates(subset=["dedup_key", "answer"]).copy()
    n_after_exact = len(df_out)

    df_out = df_out.reset_index(drop=True)

    dedup_stats = {
        "before": n_before,
        "removed_conflicting_duplicates": n_before - n_after_conflicts,
        "removed_exact_duplicates": n_after_conflicts - n_after_exact,
        "after": len(df_out),
    }

    return df_out, dedup_stats

In [27]:
df, stats = build_clean_dataframe(
    raw_records=raw_records,
    section_filter_fn=section_is_kept,
    source_name="quaderno_prefettizia",
)

print("Raw rows:", len(raw_records))
print("Main legal after malformed removal:", len(df))
print("\nRemoval stats:")
for key, value in sorted(stats.items()):
    print(f"{key}: {value}")

df, dedup_stats = deduplicate_dataframe(df)

print("\nDedup stats:")
for key, value in dedup_stats.items():
    print(f"{key}: {value}")

display(df.head())

Raw rows: 4525
Main legal after malformed removal: 3189

Removal stats:
removed_bad_question: 1
removed_duplicate_options: 2
removed_section:ECONOMIA POLITICA - DIFFICOLTA' MEDIA: 1333

Dedup stats:
before: 3189
removed_conflicting_duplicates: 0
removed_exact_duplicates: 0
after: 3189


,source_idx,section,section_base,question,options,prompt,answer,metadata,dedup_key
0,0,DIRITTO COMUNITARIO - DIFFICOLTA' FACILE,DIRITTO COMUNITARIO,"In materia di adesione all'unione Europea, sec...",{'A': 'non essere coinvolto dalla procedura di...,"In materia di adesione all'unione Europea, sec...",D,"{'source': 'quaderno_prefettizia', 'section': ...",in materia di adesione allunione europea secon...
1,1,DIRITTO COMUNITARIO - DIFFICOLTA' FACILE,DIRITTO COMUNITARIO,In relazione all'art. 41 (Diritto ad una buona...,"{'A': 'Unilateralità, spontaneità, proporziona...",In relazione all'art. 41 (Diritto ad una buona...,C,"{'source': 'quaderno_prefettizia', 'section': ...",in relazione allart 41 diritto ad una buona am...
2,2,DIRITTO COMUNITARIO - DIFFICOLTA' FACILE,DIRITTO COMUNITARIO,Indicare quale tra le seguenti affermazioni su...,{'A': 'La rappresentanza del Consiglio in tutt...,Indicare quale tra le seguenti affermazioni su...,D,"{'source': 'quaderno_prefettizia', 'section': ...",indicare quale tra le seguenti affermazioni su...
3,3,DIRITTO COMUNITARIO - DIFFICOLTA' FACILE,DIRITTO COMUNITARIO,"L’articolo 30 del TFUE, nella disciplina dei d...",{'A': 'il divieto ai dazi non riguarda quelli ...,"L’articolo 30 del TFUE, nella disciplina dei d...",C,"{'source': 'quaderno_prefettizia', 'section': ...",larticolo 30 del tfue nella disciplina dei daz...
4,4,DIRITTO COMUNITARIO - DIFFICOLTA' FACILE,DIRITTO COMUNITARIO,Attraverso la formazione di un gruppo Politico...,"{'A': 'i fondi di operatività strategica', 'B'...",Attraverso la formazione di un gruppo Politico...,D,"{'source': 'quaderno_prefettizia', 'section': ...",attraverso la formazione di un gruppo politico...


In [28]:
df_economia, economia_stats = build_clean_dataframe(
    raw_records=raw_records,
    section_filter_fn=section_is_economia_politica,
    source_name="quaderno_prefettizia",
)

print("Raw rows:", len(raw_records))
print("Economia politica after malformed removal:", len(df_economia))
print("\nRemoval stats:")
for key, value in sorted(economia_stats.items()):
    print(f"{key}: {value}")

df_economia, economia_dedup_stats = deduplicate_dataframe(df_economia)

print("\nDedup stats:")
for key, value in economia_dedup_stats.items():
    print(f"{key}: {value}")

display(df_economia.head())

Raw rows: 4525
Economia politica after malformed removal: 1328

Removal stats:
removed_bad_question: 1
removed_duplicate_options: 4
removed_section:DIRITTO AMMINISTRATIVO - DIFFICOLTA' ALTA: 146
removed_section:DIRITTO AMMINISTRATIVO - DIFFICOLTA' FACILE: 221
removed_section:DIRITTO AMMINISTRATIVO - DIFFICOLTA' MEDIA: 366
removed_section:DIRITTO CIVILE - DIFFICOLTA' ALTA: 383
removed_section:DIRITTO CIVILE - DIFFICOLTA' FACILE: 226
removed_section:DIRITTO CIVILE - DIFFICOLTA' MEDIA: 373
removed_section:DIRITTO COMUNITARIO - DIFFICOLTA' ALTA: 148
removed_section:DIRITTO COMUNITARIO - DIFFICOLTA' FACILE: 223
removed_section:DIRITTO COMUNITARIO - DIFFICOLTA' MEDIA: 367
removed_section:DIRITTO COSTITUZIONALE - DIFFICOLTA' ALTA: 149
removed_section:DIRITTO COSTITUZIONALE - DIFFICOLTA' FACILE: 222
removed_section:DIRITTO COSTITUZIONALE - DIFFICOLTA' MEDIA: 368

Dedup stats:
before: 1328
removed_conflicting_duplicates: 0
removed_exact_duplicates: 0
after: 1328


,source_idx,section,section_base,question,options,prompt,answer,metadata,dedup_key
0,3192,ECONOMIA POLITICA - DIFFICOLTA' MEDIA,ECONOMIA POLITICA,Il divario (o gap) del PIL è:,{'A': 'lo scostamento tra livello del PIL di p...,Il divario (o gap) del PIL è:\n\nA) lo scostam...,D,"{'source': 'quaderno_prefettizia', 'section': ...",il divario o gap del pil è a lo scostamento tr...
1,3193,ECONOMIA POLITICA - DIFFICOLTA' MEDIA,ECONOMIA POLITICA,Con riferimento alla funzione di produzione di...,{'A': 'La produttività media cresce e decresce...,Con riferimento alla funzione di produzione di...,B,"{'source': 'quaderno_prefettizia', 'section': ...",con riferimento alla funzione di produzione di...
2,3194,ECONOMIA POLITICA - DIFFICOLTA' MEDIA,ECONOMIA POLITICA,Nel meccanismo chiamato spirale prezzi salari:,{'A': 'una minore disoccupazione comporta un'i...,Nel meccanismo chiamato spirale prezzi salari:...,A,"{'source': 'quaderno_prefettizia', 'section': ...",nel meccanismo chiamato spirale prezzi salari ...
3,3195,ECONOMIA POLITICA - DIFFICOLTA' MEDIA,ECONOMIA POLITICA,"Le decisioni dell'impresa (cosa, come e quanto...",{'A': 'Le imprese devono decidere quanto produ...,"Le decisioni dell'impresa (cosa, come e quanto...",C,"{'source': 'quaderno_prefettizia', 'section': ...",le decisioni dellimpresa cosa come e quanto pr...
4,3196,ECONOMIA POLITICA - DIFFICOLTA' MEDIA,ECONOMIA POLITICA,La curva di Phillips è un modello empirico a e...,{'A': 'la variazione del tasso di inflazione e...,La curva di Phillips è un modello empirico a e...,D,"{'source': 'quaderno_prefettizia', 'section': ...",la curva di phillips è un modello empirico a e...


In [26]:
print("Final rows:", len(df))

print("\nCounts by section:")
display(df["section"].value_counts().sort_index().to_frame("n_examples"))

print("\nAnswer distribution:")
display(df["answer"].value_counts().sort_index().to_frame("n_examples"))

print("\nNumber of options:")
display(df["options"].apply(len).value_counts().sort_index().to_frame("n_examples"))

print("\nExamples:")
for i, row in df.head(3).iterrows():
    print("=" * 80)
    print("source_idx:", row["source_idx"])
    print("section:", row["section"])
    print("answer:", row["answer"])
    print(row["prompt"])

Final rows: 3189

Counts by section:


,n_examples
section,
DIRITTO AMMINISTRATIVO - DIFFICOLTA' ALTA,146
DIRITTO AMMINISTRATIVO - DIFFICOLTA' FACILE,221
DIRITTO AMMINISTRATIVO - DIFFICOLTA' MEDIA,366
DIRITTO CIVILE - DIFFICOLTA' ALTA,381
DIRITTO CIVILE - DIFFICOLTA' FACILE,226
DIRITTO CIVILE - DIFFICOLTA' MEDIA,373
DIRITTO COMUNITARIO - DIFFICOLTA' ALTA,148
DIRITTO COMUNITARIO - DIFFICOLTA' FACILE,223
DIRITTO COMUNITARIO - DIFFICOLTA' MEDIA,366



Answer distribution:


,n_examples
answer,
A,819
B,792
C,763
D,815



Number of options:


,n_examples
options,
4,3189



Examples:
source_idx: 0
section: DIRITTO COMUNITARIO - DIFFICOLTA' FACILE
answer: D
In materia di adesione all'unione Europea, secondo l'art. 49 TUE, un paese che intende aderire deve necessariamente:

A) non essere coinvolto dalla procedura di crisi di governo
B) aver abolito la pena di morte
C) essere in democrazia da almeno 50 anni
D) essere uno Stato europeo
source_idx: 1
section: DIRITTO COMUNITARIO - DIFFICOLTA' FACILE
answer: C
In relazione all'art. 41 (Diritto ad una buona amministrazione) della Carta dei Diritti Fondamentali dell'Unione Europea, quali sono i principi che caratterizzano il trattamento delle questioni che riguardano un individuo da parte delle istituzioni dell'Unione?

A) Unilateralità, spontaneità, proporzionalità
B) Arbitrarietà, ingiustizia e ritardi
C) Imparzialità, equità e tempestività
D) Segretezza, parzialità e indifferenza
source_idx: 2
section: DIRITTO COMUNITARIO - DIFFICOLTA' FACILE
answer: D
Indicare quale tra le seguenti affermazioni sul Consiglio

## Random qualitative inspection

In [29]:
RANDOM_SEED = 42
N_EXAMPLES = 20

def print_random_examples(df_in, title, n=20, random_seed=42):
    print("=" * 120)
    print(title)
    print("=" * 120)

    n = min(n, len(df_in))
    sample_df = df_in.sample(n=n, random_state=random_seed).reset_index(drop=True)

    for i, row in sample_df.iterrows():
        print("\n" + "-" * 120)
        print(f"example: {i + 1}/{n}")
        print("source_idx:", row["source_idx"])
        print("section:", row["section"])
        print("section_base:", row.get("section_base", ""))
        print("answer:", row["answer"])
        print("n_options:", len(row["options"]))
        print()
        print(row["prompt"])


print_random_examples(
    df,
    title="Random examples: main Quaderno legal/civic subset",
    n=N_EXAMPLES,
    random_seed=RANDOM_SEED,
)

print_random_examples(
    df_economia,
    title="Random examples: Economia Politica subset",
    n=N_EXAMPLES,
    random_seed=RANDOM_SEED,
)

Random examples: main Quaderno legal/civic subset

------------------------------------------------------------------------------------------------------------------------
example: 1/20
source_idx: 1030
section: DIRITTO AMMINISTRATIVO - DIFFICOLTA' MEDIA
section_base: DIRITTO AMMINISTRATIVO
answer: D
n_options: 4

Che cosa si intende con la sigla SPID?

A) Sistema di Pubblicazione per Istanze e Documenti
B) Servizio Pubblico per l'Integrazione dei Dati
C) Servizio Pubblico di Identificazione Digitale
D) Sistema Pubblico di Identità Digitale

------------------------------------------------------------------------------------------------------------------------
example: 2/20
source_idx: 1002
section: DIRITTO AMMINISTRATIVO - DIFFICOLTA' MEDIA
section_base: DIRITTO AMMINISTRATIVO
answer: A
n_options: 4

Ai sensi della legge n. 689/1981, la violazione amministrativa, quando é possibile, deve essere contestata immediatamente:

A) tanto al trasgressore quanto alla persona che sia obbligata 

## Build final records

In [30]:
def make_records_from_df(df_in, dataset_name, id_prefix):
    records_out = []

    for idx, row in df_in.iterrows():
        record = {
            "id": f"{id_prefix}_italian_{idx}",
            "dataset": dataset_name,
            "language": LANGUAGE,
            "split": SPLIT,
            "task_type": "mcq",
            "prompt": row["prompt"],
            "answer": row["answer"],
            "metadata": row["metadata"],
            "source_id": f"quaderno_prefettizia_{row['source_idx']}",
        }
        records_out.append(record)

    return records_out


records = make_records_from_df(
    df_in=df,
    dataset_name=MAIN_DATASET,
    id_prefix="quaderno_prefettizia_clean",
)

economia_records = make_records_from_df(
    df_in=df_economia,
    dataset_name=ECONOMIA_DATASET,
    id_prefix="quaderno_prefettizia_economia_politica_clean",
)

print("Built main records:", len(records))
print("Built economia politica records:", len(economia_records))

print("\nMain example:")
print(json.dumps(records[0], ensure_ascii=False, indent=2))

print("\nEconomia politica example:")
print(json.dumps(economia_records[0], ensure_ascii=False, indent=2))

Built main records: 3189
Built economia politica records: 1328

Main example:
{
  "id": "quaderno_prefettizia_clean_italian_0",
  "dataset": "quaderno_prefettizia_clean",
  "language": "italian",
  "split": "all",
  "task_type": "mcq",
  "prompt": "In materia di adesione all'unione Europea, secondo l'art. 49 TUE, un paese che intende aderire deve necessariamente:\n\nA) non essere coinvolto dalla procedura di crisi di governo\nB) aver abolito la pena di morte\nC) essere in democrazia da almeno 50 anni\nD) essere uno Stato europeo",
  "answer": "D",
  "metadata": {
    "source": "quaderno_prefettizia",
    "section": "DIRITTO COMUNITARIO - DIFFICOLTA' FACILE",
    "original_id": "quaderno_prefettizia_italian_0",
    "section_base": "DIRITTO COMUNITARIO",
    "source_idx": 0,
    "n_options": 4
  },
  "source_id": "quaderno_prefettizia_0"
}

Economia politica example:
{
  "id": "quaderno_prefettizia_economia_politica_clean_italian_0",
  "dataset": "quaderno_prefettizia_economia_politica_c

In [31]:
def validate_and_save(records_in, output_path, expected_dataset):
    ids = [record["id"] for record in records_in]
    assert len(ids) == len(set(ids)), "Duplicate IDs found."

    prompt_keys = set()
    for record in records_in:
        validate_record(record, expected_dataset=expected_dataset)
        key = normalize_for_dedup(record["prompt"])
        assert key not in prompt_keys, "Duplicate prompt found."
        prompt_keys.add(key)

    write_jsonl(records_in, output_path)
    print(f"Saved {len(records_in)} records to:")
    print(output_path)

    loaded = read_jsonl(output_path)
    assert len(loaded) == len(records_in)

    for record in loaded:
        validate_record(record, expected_dataset=expected_dataset)

    print("Reload validation passed.")


validate_and_save(
    records_in=records,
    output_path=OUTPUT_PATH,
    expected_dataset=MAIN_DATASET,
)

validate_and_save(
    records_in=economia_records,
    output_path=ECONOMIA_OUTPUT_PATH,
    expected_dataset=ECONOMIA_DATASET,
)

Saved 3189 records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/quaderno_prefettizia_clean/quaderno_prefettizia_clean.jsonl
Reload validation passed.
Saved 1328 records to:
/scratch/workspace/emma/standard-project-m2-mystery-machine/multilingual/data/processed/quaderno_prefettizia_clean/quaderno_prefettizia_economia_politica_clean.jsonl
Reload validation passed.
